# Predicting Wine Quality Using Machine Learning
- *Project by*: Genevieve Orr, Anna Phipps, Ellie Roberts, Hayden Gamble
- *Course*: ENGR:3110 Introduction to AI and Machine Learning

## Introduction:
Our goal for this project is to develop and evaluate different machine learning regression models to predict the quality of wine based on its physiochemical properties. The dataset contains 4898 wine samples and 11 input features. We will test the different models on a training set to determine which one works best at predicting wine quality.

## Methods: Data
- Our project uses **regression** to predict a wine's numerical quality score based on its physicochemical properties. The quality scores are treated as numerical values rather than separate categories.

- We used the [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset, specifically the white wine data, containing `4,898 samples` with `11 numerical features` describing the wine's physicochemical properties. Our target variable is the wine's quality score.

- We loaded the dataset using Pandas and checked for missing values and duplicate rows. No missing values were found, but `937 duplicate rows` were removed, leaving `3,961 samples`. We also examined summary statistics and plotted the distribution of quality scores.

- Finally, we separated the input features from the target variable and split the cleaned dataset into `60% training`, `20% validation`, and `20% testing`. We used stratified splitting to maintain similar quality-score distributions across the sets and random_state=42 for reproducibility. The test set will be reserved for evaluating the final selected model.


In [ ]:
## imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import seaborn as sns
from sklearn import (datasets, neighbors,
                     naive_bayes,
                     model_selection as skms,
                     linear_model, metrics)
from pathlib import Path

# Load data and create the shared splits - Hayden
csv_path = Path("winequality-white.csv")
wine_df = pd.read_csv(csv_path, sep=";")

print("Original dataset shape (rows, columns):", wine_df.shape)
print("Number of missing values:", wine_df.isna().sum().sum())
print("Exact duplicate rows:", wine_df.duplicated().sum())
print("Data types:\n", wine_df.dtypes)
display(wine_df.head())

# Remove identical full rows so the same observation cannot appear in both training and testing
wine_df = wine_df.drop_duplicates().reset_index(drop=True)
print("\nDataset shape after removing duplicate rows:", wine_df.shape)

# Descriptive summary and distribution of recorded quality scores.
display(wine_df.describe().T)
quality_counts = wine_df["quality"].value_counts().sort_index()
print("\nWine quality counts after cleaning:\n", quality_counts)

fig, ax = plt.subplots(figsize=(7, 4))
quality_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribution of white wine quality scores")
ax.set_xlabel("Quality score")
ax.set_ylabel("Number of wine samples")
plt.tight_layout()
plt.show()

# Separate the 11 feature columns from the quality target.
X = wine_df.drop(columns=["quality"])
y = wine_df["quality"]

# First reserve 20% for the untouched final test set.
X_train_val, X_test, y_train_val, y_test = skms.train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# From the remaining 80%, use 25% for validation (20% of all rows).
# Final proportions: 60% train, 20% validation, 20% test.
X_train, X_val, y_train, y_val = skms.train_test_split(
    X_train_val, y_train_val, test_size=0.25,
    random_state=42, stratify=y_train_val
)

print("\nFinal data split sizes:")
print(f"Training:   {len(X_train)} rows, {X_train.shape[1]} features")
print(f"Validation: {len(X_val)} rows, {X_val.shape[1]} features")
print(f"Testing:    {len(X_test)} rows, {X_test.shape[1]} features")


## Methods: Training/Validation
- Linear Regression
    - All Features
        - ```Text here```
    - Reduced Feature Set
        - ```Text here```
- KNN Regression
    - `k = 3`
        - We use *K-Nearest Neighbors Regression* with `k = 3` to predict wine quality from the average quality score of the three most similar training wines. Since KNN uses distances, we standardize the 11 features with `StandardScaler` in a pipeline fitted only on the training data. We then evaluate the model on the validation set using RMSE, MAE, and R². The test set is not used in this step.
    - `k = 7`
        - ```Text here```
    - `k = 11`
        - We use K-Nearest Neighbors Regression with k = 11 to predict wine quality based on the 11 physiochemical features in our dataset. The model identifies the 11 wines with the most similar characteristics and uses the most common quality rating to make its prediction. We then evaluate the model on the validation set using RMSE and compare it with other models.

In [ ]:
# use training set to evaluate models
# Linear regression (all features) - Anna



In [ ]:
# Linear regression (reduced feature set) - Anna


In [ ]:
# KNN Regression (k=3) - Hayden
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# Scaling is learned from X_train only; X_val is transformed using that same scaler.
knn3_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=3)
)

# Fit on the training set and predict wine quality on the validation set.
knn3_model.fit(X_train, y_train)
knn3_val_predictions = knn3_model.predict(X_val)

# Regression metrics for comparison with the other models (lower RMSE/MAE is better).
knn3_val_rmse = np.sqrt(metrics.mean_squared_error(y_val, knn3_val_predictions))
knn3_val_mae = metrics.mean_absolute_error(y_val, knn3_val_predictions)
knn3_val_r2 = metrics.r2_score(y_val, knn3_val_predictions)

knn3_results = {
    "Model": "KNN Regression (k=3)",
    "Validation RMSE": knn3_val_rmse,
    "Validation MAE": knn3_val_mae,
    "Validation R2": knn3_val_r2,
}
display(pd.DataFrame([knn3_results]).round(3))


In [ ]:
# kNN (k=7) - Ellie

# Fitting scaler and model for training
knn7_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=7)
)

# Fit on the training set and predict wine quality on the validation set.
knn7_model.fit(X_train, y_train)
knn7_val_predictions = knn7_model.predict(X_val)

# Validating metrics
knn7_val_rmse = np.sqrt(metrics.mean_squared_error(y_val,knn7_val_predictions))
knn7_val_r2 = metrics.r2_score(y_val, knn7_val_predictions)
knn7_val_mae = metrics.mean_absolute_error(y_val, knn7_val_predictions)

knn7_results = {"Model": "KNN Regression (k=7)", "Validation RMSE": knn7_val_rmse,}
display(pd.DataFrame([knn7_results]).round(3))


In [ ]:
# kNN (k=11) - Gen

# create model
knn11_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=11))

# fit model using training date and make predictions
knn11_model.fit(X_train, y_train)
knn11_val_predictions = knn11_model.predict(X_val)

# calculate rmse
knn11_val_rmse = np.sqrt(metrics.mean_squared_error(y_val, knn11_val_predictions))

# print results
print("K-Nearest Neighbor (k = 11) RMSE:", round(knn11_val_rmse, 3))


## Determining Best Model


In [ ]:
# Comparing validation accuracies in models - Ellie
model_results = [
    #linear regression(all),
    #linear regression(reduced),
    knn3_results,         # kNN 3 neighbor results
    knn7_results,         # kNN 7 neighbor results
    knn11_val_rmse        # kNN 11 neighbor results
]

# Comparing all models sorting by lowest RMSE (best models are first)
comparison_df = pd.DataFrame(model_results).sort_values(by="Validation RMSE", ascending=True).reset_index(drop=True)
display(comparison_df.round(3))

## Results on Test Set
We evaluated our final model using the 20% of the dataset reserved for testing, which was not used during model training or selection. After fitting the best performing model on the training data, we used it to predict wine quality scores for the test set. We then calculated the Root Mean Squared Error (RMSE) by comparing the predicted quality scores to the actual scores to measure the model's prediction error.

In [ ]:
# use test set to test final model - Gen
# fit model to training data
best_model.fit(X_train, y_train)

# model predictions using test data
test_predictions = best_model.predict(X_test)

# calculate rmse
final_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_predictions))

# print results
print("Final Test RMSE:", round(final_rmse, 3))

## Discussion/Conclusion


## Disclosures